# Semantic search over a small corpus with LEAF

What sentence embeddings are *actually* for: matching queries to documents by **meaning**, not keywords. LEAF (`mdbr-leaf-ir`) was trained for retrieval, so the canonical pipeline is:

1. Embed the corpus once → vectors
2. Embed each query → vector
3. Rank corpus entries by cosine similarity to the query vector

This notebook fits all of that in four cells against an in-memory FAQ corpus. The interesting case: queries that share **no content words** with the right answer still rank it #1.

Uses the Spring-AI-shaped `EmbeddingModel.call(EmbeddingRequest)` from `sk.ainet.llm.api`. Self-contained — pulls `skainet-transformers` from Maven Central, no project classpath needed.

**Versions are BOM-aligned.** Each artifact is pinned to the version `sk.ainet.transformers:skainet-transformers-bom:0.23.5` would resolve it to (the BOM itself can't go in `@file:DependsOn` — BOMs ship POM-only).

## 1. Dependencies, imports, and model

In [1]:
@file:DependsOn(
    "sk.ainet.transformers:skainet-transformers-providers:0.40.2"
)


org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: Dependency resolution failed: Failed to resolve [Dependency(value=sk.ainet.transformers:skainet-transformers-providers:0.40.2, hasSources=MAYBE)]:
File 'sk.ainet.transformers:skainet-transformers-providers:0.40.2' not found
AmperDependencyResolutionException: Dependency 'sk.ainet.core:skainet-lang-core:unspecified' was not resolved

In [ ]:
import kotlinx.coroutines.runBlocking
import sk.ainet.data.source.DataSourceRequest
import sk.ainet.data.source.JvmDataSourceResolver
import java.nio.file.Files
import java.nio.file.Path
import java.util.zip.ZipInputStream
import kotlin.io.path.exists
import kotlin.io.path.fileSize

val gloveDir = Path.of(
    System.getProperty("user.home"),
    ".deliverance",
    "glove"
)
val gloveFile = gloveDir.resolve("glove.6B.s.txt")

if (!gloveFile.exists()) {
    Files.createDirectories(gloveDir)

    // SKaiNET: HF download + cache
    val gloveZip = runBlocking {
        JvmDataSourceResolver().resolve(
            DataSourceRequest("hf://MongoDB/mdbr-leaf-ir/")
        )
    }

    val zipFile = Path.of(requireNotNull(gloveZip.localPath))

    ZipInputStream(Files.newInputStream(zipFile)).use { zip ->
        var entry = zip.nextEntry

        while (entry != null) {
            if (entry.name == "glove.6B.50d.txt") {
                Files.newOutputStream(gloveFile).use { out ->
                    zip.copyTo(out)
                }
                break
            }
            entry = zip.nextEntry
        }
    }

    require(gloveFile.exists()) {
        "glove.6B.50d.txt not found in glove.6B.zip"
    }

    println("Saved to $gloveFile (${gloveFile.fileSize() / 1024 / 1024} MB)")
} else {
    println("Already cached at $gloveFile (${gloveFile.fileSize() / 1024 / 1024} MB)")
}

In [ ]:
@file:Repository("https://repo1.maven.org/maven2")
@file:DependsOn("sk.ainet.transformers:skainet-transformers-providers:0.23.5")
@file:DependsOn("sk.ainet.transformers:skainet-transformers-inference-bert:0.23.5")
@file:DependsOn("sk.ainet.core:skainet-lang-core:0.23.1")
@file:DependsOn("sk.ainet.core:skainet-io-core:0.23.1")
@file:DependsOn("sk.ainet.core:skainet-io-safetensors:0.23.1")
@file:DependsOn("sk.ainet.core:skainet-backend-cpu:0.23.1")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core:1.10.2")

import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.io.JvmRandomAccessSource
import sk.ainet.io.safetensors.SafeTensorsParametersLoader
import sk.ainet.lang.types.FP32
import sk.ainet.llm.api.EmbeddingModel
import sk.ainet.llm.api.EmbeddingRequest
import sk.ainet.llm.api.EmbeddingResponse
import sk.ainet.llm.providers.SkaiNetEmbeddingModel
import sk.ainet.models.bert.BertRuntime
import sk.ainet.models.bert.HuggingFaceTokenizer
import sk.ainet.models.bert.MDBR_LEAF_IR_CONFIG
import sk.ainet.models.bert.loadBertWeights
import kotlinx.coroutines.runBlocking
import java.nio.file.Path
import kotlin.io.path.Path
import kotlin.io.path.exists
import kotlin.io.path.readText
import kotlin.math.sqrt

fun loadLeafModel(modelDir: Path, ctx: DirectCpuExecutionContext): EmbeddingModel {
    val tokenizer = HuggingFaceTokenizer.fromVocabTxt(modelDir.resolve("vocab.txt").readText())
    val baseWeights = sequenceOf("model.safetensors", "pytorch_model.safetensors")
        .map { modelDir.resolve(it) }
        .firstOrNull { it.exists() }
        ?: error("No safetensors file in $modelDir")
    val denseWeights = modelDir.resolve("2_Dense/model.safetensors").takeIf { it.exists() }
    val config = if (denseWeights != null) MDBR_LEAF_IR_CONFIG else MDBR_LEAF_IR_CONFIG.copy(projectionDim = null)
    val loaders = listOfNotNull(baseWeights, denseWeights).map { file ->
        SafeTensorsParametersLoader(
            sourceProvider = { JvmRandomAccessSource.open(file.toString()) },
            onProgress = { _, _, _ -> },
        )
    }
    val weights = runBlocking { loadBertWeights(loaders, ctx, FP32::class, config) }
    val runtime = BertRuntime(ctx, weights, FP32::class)
    return SkaiNetEmbeddingModel(
        runtime = runtime,
        tokenizer = tokenizer,
        dimensions = config.projectionDim ?: config.hiddenSize,
        modelId = modelDir.fileName.toString(),
    )
}

val ctx = DirectCpuExecutionContext()
val modelDir = Path("${System.getProperty("user.home")}/.deliverance/MongoDB_mdbr-leaf-ir")
val model: EmbeddingModel = loadLeafModel(modelDir, ctx)

println("dimensions = ${model.dimensions}")

## 2. Embed the corpus in one batched request

A small support-FAQ corpus. One `model.call(EmbeddingRequest(...))` per *batch*, not per sentence — that's the Spring-AI shape, and the right pattern when you index a static document set.

In [ ]:
val corpus = listOf(
    "We accept Visa, Mastercard, and Amex for monthly and annual subscriptions.",
    "If you forgot your password, click 'Forgot password' on the login page; the reset link expires after 30 minutes.",
    "Two-factor authentication is configured under Settings > Security and supports TOTP apps like Google Authenticator.",
    "Annual subscriptions are eligible for a prorated refund within 14 days of renewal.",
    "Our REST API lives at api.example.com — see the developer docs for OAuth client setup and rate limits.",
    "Cancellations take effect at the end of the current billing period; you keep access until then.",
    "Our offices are open Monday through Friday, 9am to 6pm Central European Time.",
    "We support 12 languages in the UI including English, German, French, Spanish, Japanese, and Mandarin.",
)

val response: EmbeddingResponse = model.call(EmbeddingRequest(corpus))
val corpusVectors: List<FloatArray> = response.embeddings.sortedBy { it.index }.map { it.vector }

println("corpus size:    ${corpus.size}")
println("prompt tokens:  ${response.usage?.promptTokens}")
println("vector dim:     ${corpusVectors.first().size}")

## 3. A `search(query)` function over the in-memory index

Single-query path uses the `embed(text): FloatArray` convenience overload (it's `call(...)` under the hood, just without the request/response wrapping). Plain cosine math against the cached corpus vectors — no tensor library needed at this size.

In [ ]:
data class Hit(val score: Float, val text: String)

fun cosine(a: FloatArray, b: FloatArray): Float {
    var dot = 0f; var na = 0f; var nb = 0f
    for (i in a.indices) { dot += a[i] * b[i]; na += a[i] * a[i]; nb += b[i] * b[i] }
    return dot / (sqrt(na) * sqrt(nb))
}

fun search(query: String, topK: Int = 3): List<Hit> {
    val q = model.embed(query)
    return corpus.zip(corpusVectors)
        .map { (text, vec) -> Hit(cosine(q, vec), text) }
        .sortedByDescending { it.score }
        .take(topK)
}

fun show(query: String) {
    println("Q: $query")
    search(query).forEachIndexed { i, h ->
        val preview = if (h.text.length > 80) h.text.take(77) + "..." else h.text
        println("  #${i + 1}  ${"%.4f".format(h.score)}   $preview")
    }
    println()
}

## 4. Three queries that show what's interesting

1. **Literal match** — easy case, validates the plumbing.
2. **Paraphrase with zero content-word overlap** — keyword search would miss this; embeddings nail it.
3. **Negation / discrimination** — semantically related to several entries, must pick the right one.

In [ ]:
show("How do I reset my password?")
show("How can I pay for my plan?")
show("What happens to my account when I cancel?")

## Why this beats keyword search

Look at query #2: `"How can I pay for my plan?"` shares **no content words** with the right answer (`"We accept Visa, Mastercard, and Amex…"`). A BM25 / TF-IDF index would rank that pairing near zero. Sentence embeddings rank it #1 because the model learned during training that *pay* / *card brand* / *subscription* / *plan* co-occur in semantically similar contexts.

Query #3 (`"What happens to my account when I cancel?"`) is interesting in the other direction — it's plausibly close to the refund entry, the password-reset entry ("keep access"), or the cancellation entry. The model has to discriminate between *related* meanings, not just measure broad topical similarity. Cosine ranking over LEAF embeddings gets it right.

**Production-style equivalent in this repo:** `IndexCommand` and `AskCommand` in `sk.ainet.apps.leaf.cli` do exactly this — same model, same cosine-rank approach — but persist the corpus to a JSON-backed `VectorRepository` instead of holding it in memory, and walk a directory of markdown rather than a hardcoded list. The notebook is the inner loop; the CLI is the wrapper.

**Scaling beyond N≈100:** the per-query `O(N)` cosine sweep here is fine for a notebook. For real corpora, swap `JsonFileVectorRepository` for an ANN index (HNSW, FAISS, Qdrant, Chroma) — the `VectorRepository` interface in this project was designed exactly for that swap.

**Going multilingual:** this is the IR variant of LEAF. The MT variant (`mdbr-leaf-mt`, with `2_Dense/`) is a sibling model designed for cross-lingual retrieval — same notebook works against it, the loader auto-detects the dense layer.